In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Nastavení voleb

<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vytvořen s použitím následujících požadavků.
Doporučujeme použít tyto nebo novější verze.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Pomocí voleb můžeš přizpůsobit primitiva Estimator a Sampler. Tato část se zaměřuje na to, jak nastavit volby primitivů Qiskit Runtime. Zatímco rozhraní metody `run()` primitivů je společné pro všechny implementace, jejich volby společné nejsou. Informace o volbách [`qiskit.primitives`](https://docs.quantum.ibm.com/api/qiskit/primitives#primitives) a [`qiskit_aer.primitives`](https://qiskit.github.io/qiskit-aer/apidocs/aer_primitives.html) najdeš v příslušných referenčních dokumentacích API.

Poznámky ke specifikaci voleb v primitivech:

- `SamplerV2` a `EstimatorV2` mají oddělené třídy voleb. Dostupné volby můžeš zobrazit a aktualizovat jejich hodnoty během inicializace primitiva i po ní.
- Ke změnám atributu `options` použij metodu `update()`.
- Pokud pro volbu nezadáš žádnou hodnotu, dostane speciální hodnotu `Unset` a použijí se výchozí hodnoty serveru.
- Atribut `options` je Python typ `dataclass`. K jeho převodu na slovník můžeš použít vestavěnou metodu `asdict`.

<span id="pass-options"></span>
## Nastavení voleb primitiva
Volby můžeš nastavit při inicializaci primitiva, po jeho inicializaci nebo v metodě `run()`. Přečti si část o [pravidlech přednosti](runtime-options-overview#options-precedence), abys pochopil/a, co se stane, když je stejná volba zadána na více místech.

### Inicializace primitiva
Při inicializaci primitiva můžeš předat instanci třídy voleb nebo slovník – primitiv si z nich vytvoří kopii. Změna původního slovníku nebo instance voleb tedy neovlivní volby vlastněné primitivem.

#### Třída voleb
Při vytváření instance třídy `EstimatorV2` nebo `SamplerV2` můžeš předat instanci třídy voleb. Tyto volby se pak použijí při provádění výpočtu metodou `run()`. Volby zadávej v tomto formátu: `options.option.sub-option.sub-sub-option = choice`. Například: `options.dynamical_decoupling.enable = True`

Příklad:

`SamplerV2` a `EstimatorV2` mají oddělené třídy voleb ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) a [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)).

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

options = EstimatorOptions(
    resilience_level=2,
    resilience={"zne_mitigation": True, "zne": {"noise_factors": [1, 3, 5]}},
)

# or...
options = EstimatorOptions()
options.resilience_level = 2
options.resilience.zne_mitigation = True
options.resilience.zne.noise_factors = [1, 3, 5]

estimator = Estimator(mode=backend, options=options)

#### Slovník
Při inicializaci primitiva můžeš zadat volby jako slovník.

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

# Setting options during primitive initialization
estimator = Estimator(
    backend,
    options={
        "resilience_level": 2,
        "resilience": {
            "zne_mitigation": True,
            "zne": {"noise_factors": [1, 3, 5]},
        },
    },
)

### Aktualizace voleb po inicializaci
Volby můžeš zadat ve formátu `primitive.options.option.sub-option.sub-sub-option = choice` a využít tak automatické doplňování, nebo použít metodu `update()` pro hromadné změny.

Třídy voleb `SamplerV2` a `EstimatorV2` ([`EstimatorOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-estimator-options) a [`SamplerOptions`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-sampler-options)) není nutné instanciovat, pokud nastavuješ volby až po inicializaci primitiva.

In [3]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

# Setting options after primitive initialization
# This uses auto-complete.
estimator.options.default_shots = 4000
# This does bulk update.
estimator.options.update(
    default_shots=4000, resilience={"zne_mitigation": True}
)

<span id="run-method"></span>
### Metoda Run()
Do metody `run()` lze předat pouze hodnoty definované v rozhraní, tedy `shots` pro Sampler a `precision` pro Estimator. Tím se přepíše jakákoli hodnota nastavená pro `default_shots` nebo `default_precision` pro aktuální spuštění.

In [4]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)

sampler = Sampler(mode=backend)
# Default shots to use if not specified in run()
sampler.options.default_shots = 500
# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

# Sample two circuits with different numbers of shots.
# 100 shots is used for transpiled1 and 200 for transpiled.
sampler.run([(transpiled1, None, 100), (transpiled2, None, 200)])

<RuntimeJobV2('d5k96cn853es738djikg', 'sampler')>

### Special cases

#### Resilience level (Estimator only)

The resilience level is not actually an option that directly impacts the primitive query, but specifies a base set of curated options to build off of. In general, level 0 turns off all error mitigation, level 1 turns on options for measurement error mitigation, and level 2 turns on options for gate and measurement error mitigation.

Any options you manually specify in addition to the resilience level are applied on top of the base set of options defined by the resilience level. Therefore, in principle, you could set the resilience level to 1, but then turn off measurement mitigation, although this is not advised.

In the following example, setting the resilience level to 0 initially turns off `zne_mitigation`, but `estimator.options.resilience.zne_mitigation = True` overrides the relevant setup from `estimator.options.resilience_level = 0`.

In [5]:
from qiskit_ibm_runtime import EstimatorV2, QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = EstimatorV2(backend)

estimator.options.default_shots = 100
estimator.options.resilience_level = 0
estimator.options.resilience.zne_mitigation = True

### Zvláštní případy
#### Úroveň odolnosti (pouze Estimator)
Úroveň odolnosti ve skutečnosti není volba, která by přímo ovlivňovala dotaz na primitiv, ale určuje základní sadu přednastavených voleb, od nichž se odvíjí. Obecně platí, že úroveň 0 vypne veškerou chybovou mitigaci, úroveň 1 zapne volby pro mitigaci chyb měření a úroveň 2 zapne volby pro mitigaci chyb Gate a měření.

Veškeré volby, které zadáš ručně nad rámec úrovně odolnosti, se aplikují na vrcholu základní sady voleb definované touto úrovní. V principu tedy můžeš nastavit úroveň odolnosti na 1 a zároveň vypnout mitigaci měření, i když to není doporučeno.

V následujícím příkladu nastavení úrovně odolnosti na 0 nejprve vypne `zne_mitigation`, ale `estimator.options.resilience.zne_mitigation = True` přepíše příslušné nastavení z `estimator.options.resilience_level = 0`.

In [6]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2 as Sampler
from qiskit.circuit.library import random_iqp
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

circuit1 = random_iqp(3)
circuit1.measure_all()
circuit2 = random_iqp(3)
circuit2.measure_all()

pass_manager = generate_preset_pass_manager(
    optimization_level=3, backend=backend
)

transpiled1 = pass_manager.run(circuit1)
transpiled2 = pass_manager.run(circuit2)


# Setting shots during primitive initialization
sampler = Sampler(mode=backend, options={"default_shots": 4096})

# Setting options after primitive initialization
# This uses auto-complete.
sampler.options.default_shots = 2000

# This does bulk update.  The value for default_shots is overridden if you specify shots with run() or in the PUB.
sampler.options.update(
    default_shots=1024, dynamical_decoupling={"sequence_type": "XpXm"}
)

# Sample two circuits at 128 shots each.
sampler.run([transpiled1, transpiled2], shots=128)

<RuntimeJobV2('d5k96icjt3vs73ds5t0g', 'sampler')>

#### Shots (pouze Sampler)
Metoda `SamplerV2.run` přijímá dva argumenty: seznam PUBů, z nichž každý může určit hodnotu shots specifickou pro daný PUB, a klíčový argument shots. Tyto hodnoty shots jsou součástí rozhraní pro spouštění Sampleru a jsou nezávislé na volbách Runtime Sampleru. Mají přednost před jakýmikoli hodnotami zadanými jako volby, aby byl splněn kontrakt abstrakce Sampleru.

Pokud však `shots` není zadáno ani v žádném PUBu ani jako klíčový argument metody run (nebo jsou všechny `None`), použije se hodnota shots z voleb, zejména `default_shots`.

Shrnutí pořadí přednosti pro specifikaci shots v Sampleru pro libovolný PUB:

1. Pokud PUB specifikuje shots, použije se tato hodnota.
2. Pokud je klíčový argument `shots` zadán v `run`, použije se tato hodnota.
3. Pokud jsou `num_randomizations` a `shots_per_randomization` zadány jako volby `twirling`, počet shots je součinem těchto hodnot.
3. Pokud je zadáno `sampler.options.default_shots`, použije se tato hodnota.

Pokud jsou tedy shots zadány na všech možných místech, použije se ta s nejvyšší prioritou (shots zadané v PUBu).

#### Precision (pouze Estimator)
Precision je analogická k shots popsaným v předchozí části, s tím rozdílem, že volby Estimatoru obsahují jak `default_shots`, tak `default_precision`. Navíc, protože je ve výchozím nastavení povoleno twirling Gate, součin `num_randomizations` a `shots_per_randomization` má přednost před těmito dvěma volbami.

Konkrétně pro libovolný PUB Estimatoru:

1. Pokud PUB specifikuje precision, použije se tato hodnota.
2. Pokud je klíčový argument precision zadán v `run`, použije se tato hodnota.
2. Pokud jsou `num_randomizations` a `shots_per_randomization` zadány jako [volby `twirling`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options) (ve výchozím nastavení povoleno), použije se jejich součin k řízení množství dat.
3. Pokud je zadáno `estimator.options.default_shots`, použije se tato hodnota k řízení množství dat.
4. Pokud je zadáno `estimator.options.default_precision`, použije se tato hodnota.

Pokud je například precision zadáno na všech čtyřech místech, použije se ta s nejvyšší prioritou (precision zadaná v PUBu).

> **Note:** Precision se škáluje nepřímo úměrně s využitím. Čím nižší je precision, tím více času QPU je potřeba ke spuštění.

## Často používané volby
K dispozici je mnoho voleb, ale nejčastěji se používají následující:

<span id="shots"></span>
### Shots
U některých algoritmů je nastavení konkrétního počtu shots klíčovou součástí jejich postupu. Shots (nebo precision) lze zadat na více místech. Priorita je následující:

Pro libovolný PUB Sampleru:

1. Celočíselné shots obsažené v PUBu
2. Hodnota `run(...,shots=val)`
3. Hodnota `options.default_shots`

Pro libovolný PUB Estimatoru:

1. Desetinná precision obsažená v PUBu
2. Hodnota `run(...,precision=val)`
3. Hodnota `options.default_shots`
4. Hodnota `options.default_precision`

Příklad:

In [7]:
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import EstimatorV2 as Estimator

service = QiskitRuntimeService()
backend = service.least_busy(operational=True, simulator=False)

estimator = Estimator(mode=backend)

estimator.options.max_execution_time = 2500

<span id="no-error-mitigation"></span>
## Turn off all error mitigation and error suppression

You can turn off all error mitigation and suppression if you are, for example, doing research on your own mitigation techniques. To accomplish this, for EstimatorV2, set `resilience_level = 0`. For SamplerV2, no changes are necessary because no error mitigation or suppression options are enabled by default.

Example:

Turn off all error mitigation and suppression in Estimator.

In [8]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator, QiskitRuntimeService

# Define the service.  This allows you to access IBM QPU.
service = QiskitRuntimeService()

# Get a backend
backend = service.least_busy(operational=True, simulator=False)

# Define Estimator
estimator = Estimator(backend)

options = estimator.options

# Turn off all error mitigation and suppression
options.resilience_level = 0

### Maximální doba spuštění
Maximální doba spuštění (`max_execution_time`) omezuje, jak dlouho může úloha běžet. Pokud úloha překročí tento časový limit, bude nuceně zrušena. Tato hodnota se vztahuje na jednotlivé úlohy bez ohledu na to, zda běží v režimu job, session nebo batch.

Hodnota je nastavena v sekundách na základě kvantového času (nikoli reálného času), což je doba, po kterou je QPU věnována zpracování tvé úlohy. Při použití režimu lokálního testování se tato hodnota ignoruje, protože tento režim kvantový čas nevyužívá.